<a href="https://colab.research.google.com/github/ashishkunche9-lab/blockchain-projects/blob/main/02-merkle-tree/merkle_tree.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import hashlib, json, random

def h(x):
    return hashlib.sha256(x.encode()).hexdigest()

def leaf_hash(record):
    return h(json.dumps(record, sort_keys=True))

def build_tree(records):
    level = [leaf_hash(r) for r in records]
    tree = [level]
    while len(level) > 1:
        if len(level) % 2 == 1:
            level = level + [level[-1]]
        level = [h(level[i] + level[i + 1]) for i in range(0, len(level), 2)]
        tree.append(level)
    return tree

def get_proof(records, index):
    level = [leaf_hash(r) for r in records]
    proof = []
    while len(level) > 1:
        if len(level) % 2 == 1:
            level = level + [level[-1]]
        sib = index ^ 1
        proof.append((level[sib], "right" if sib > index else "left"))
        level = [h(level[i] + level[i + 1]) for i in range(0, len(level), 2)]
        index //= 2
    return proof

def verify_proof(record, proof, root):
    cur = leaf_hash(record)
    for sib, side in proof:
        cur = h(cur + sib) if side == "right" else h(sib + cur)
    return cur == root

# Simulated turbine alarm records
random.seed(1)
names = ["Vibration (mm/s)", "Bearing temp (C)", "Lube oil pressure (bar)"]
records = [{"id": i, "alarm": random.choice(names),
            "value": round(random.uniform(1, 95), 1)} for i in range(8)]

tree = build_tree(records)
root = tree[-1][0]
print(len(records), "records")
print("Merkle root:", root)

8 records
Merkle root: 60f7595d43333ca3a3548001602483ca2a5c655a7cf39ad3ee4294fc1d29a6a8


In [2]:
import copy
import pandas as pd
from IPython.display import display

original_leaves = tree[0]

def record_table(recs):
    rows = []
    for i, r in enumerate(recs):
        ok = leaf_hash(r) == original_leaves[i]
        rows.append({"#": r["id"], "Alarm": r["alarm"], "Value": r["value"],
                     "Leaf hash": leaf_hash(r)[:8],
                     "Status": "OK" if ok else "TAMPERED"})
    return pd.DataFrame(rows)

def color(row):
    c = "#c8f7c5" if row["Status"] == "OK" else "#f7a5a5"
    return [f"background-color: {c}; color: black"] * len(row)

print("STEP 1: Original records")
display(record_table(records).style.apply(color, axis=1).hide(axis="index"))
print("Root:", build_tree(records)[-1][0][:16], "(matches saved root)")

tampered = copy.deepcopy(records)
tampered[5]["value"] = 0.5
new_root = build_tree(tampered)[-1][0]

print("\nSTEP 2: Someone edits record 5")
display(record_table(tampered).style.apply(color, axis=1).hide(axis="index"))
print("Root:", new_root[:16], "(DOES NOT match)" if new_root != root else "")

STEP 1: Original records


#,Alarm,Value,Leaf hash,Status
0,Vibration (mm/s),54.500000,d118d074,OK
1,Vibration (mm/s),25.000000,e4914426,OK
2,Bearing temp (C),72.500000,212f362a,OK
3,Bearing temp (C),62.200000,e57add15,OK
4,Vibration (mm/s),9.800000,2a755a84,OK
5,Vibration (mm/s),85.000000,1d50260b,OK
6,Bearing temp (C),41.700000,253daad1,OK
7,Vibration (mm/s),66.400000,139a8873,OK


Root: 60f7595d43333ca3 (matches saved root)

STEP 2: Someone edits record 5


#,Alarm,Value,Leaf hash,Status
0,Vibration (mm/s),54.500000,d118d074,OK
1,Vibration (mm/s),25.000000,e4914426,OK
2,Bearing temp (C),72.500000,212f362a,OK
3,Bearing temp (C),62.200000,e57add15,OK
4,Vibration (mm/s),9.800000,2a755a84,OK
5,Vibration (mm/s),0.500000,355bd8b0,TAMPERED
6,Bearing temp (C),41.700000,253daad1,OK
7,Vibration (mm/s),66.400000,139a8873,OK


Root: 1544547419e2566b (DOES NOT match)


In [3]:
def test(name, ok):
    print(f"{name:34} -> {'PASS' if ok else 'FAIL'}")

proof3 = get_proof(records, 3)

test("Valid record, valid proof", verify_proof(records[3], proof3, root))

edited = copy.deepcopy(records[3]); edited["value"] = 0.5
test("Edited record rejected", not verify_proof(edited, proof3, root))

test("Wrong proof rejected", not verify_proof(records[3], get_proof(records, 4), root))

bad_root = h("fake")
test("Wrong root rejected", not verify_proof(records[3], proof3, bad_root))

for i in range(len(records)):
    assert verify_proof(records[i], get_proof(records, i), root)
test("All 8 records verify", True)

print(f"\nProof size: {len(proof3)} hashes to check 1 record out of {len(records)}")

Valid record, valid proof          -> PASS
Edited record rejected             -> PASS
Wrong proof rejected               -> PASS
Wrong root rejected                -> PASS
All 8 records verify               -> PASS

Proof size: 3 hashes to check 1 record out of 8
